# 1️⃣ Connexion & Préparation des Données

In [8]:
import streamlit as st
from sqlalchemy import create_engine
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
db = os.getenv("DB_NAME")
port = os.getenv("DB_PORT")


engine = create_engine(
    f"postgresql://{user}:{password}@{host}:{port}/{db}"
)

st.title("FinanceCore Dashboard")
st.write("Testing connection...")

df = pd.read_sql("SELECT 1", engine)
st.success("Connexion réussie")
st.write(df)

2026-04-16 12:05:31.494 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 12:05:31.560 
  command:

    streamlit run c:\Users\user\Desktop\brief06\venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-16 12:05:31.560 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 12:05:31.560 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 12:05:31.561 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 12:05:31.561 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 12:05:31.562 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 12:05:31.628 Thread 'MainT

Calculer les métriques agrégées nécessaires à chaque page du dashboard

In [ ]:
import pandas as pd
df = pd.read_csv('financecore_clean.csv')
df['date_transaction'] = pd.to_datetime(df['date_transaction'])

print("Chargement des donnees termine. Vous pouvez maintenant calculer les Metrics.")

In [ ]:
total_transactions = df['montant_eur'].count()
taux_rejet_global = (df['rejet'].sum() / total_transactions) * 100

print(taux_rejet_global)

taux_rejet_global = round(taux_rejet_global, 2)

In [ ]:
print(f"Total Transactions: {total_transactions}")
print(f"Volume Total: {volume_total:,.2f} €")
print(f"Moyenne par Transaction: {panier_moyen:,.2f} €")
print(f"Taux de Rejet Global: {taux_rejet_global:.2f}%")

Total Transactions: 2000
Volume Total: -242,279.41 €
Moyenne par Transaction: -121.14 €
Taux de Rejet Global: 0.00%


In [ ]:
performance_agence = df.groupby('agence').agg({
    'transaction_id': 'count',
    'montant_eur': 'sum'
}).rename(columns={'transaction_id': 'nombre_trans', 'montant_eur': 'volume_total'}).reset_index()

repartition_produit = df.groupby('produit').agg({
    'montant_eur': 'sum'
}).sort_values(by='montant_eur', ascending=False).reset_index()

print("--- Top Agences ---")
print(performance_agence.head())

--- Top Agences ---
                 agence  nombre_trans  volume_total
0    Bordeaux-Meriadeck           354     -16456.04
1     Lille-Grand-Place           214      -8645.83
2        Lyon-Part-Dieu           299     -20878.00
3  Marseille-Vieux-Port           323     -20457.66
4       Nantes-Commerce           200     -60635.26


In [ ]:
analyse_risque = df.groupby('categorie_risque').agg({
    'is_anomaly': 'sum',
    'is_outlier_montant': 'sum',
    'client_id': 'nunique'
}).reset_index()

df['month_year'] = df['date_transaction'].dt.to_period('M')
anomalies_temporelles = df[df['is_anomaly'] == True].groupby('month_year').size()

print("--- Analyse des Risques ---")
print(analyse_risque)

--- Analyse des Risques ---
  categorie_risque  is_anomaly  is_outlier_montant  client_id
0             High          31                  30         32
1              Low          26                  23         39
2           Medium          55                  55        134
